# Week 17: Production RAG & Vector Search
## Notebook 08: Self-RAG: Adaptive Retrieval & Reflection Critique Tokens

### 1. Conceptual Intuition & Theoretical Background
Standard RAG systems retrieve documents indiscriminately for every query, even when retrieval is unnecessary (e.g. "Write an essay about my vacation") or harmful (retrieving noisy/conflicting passages that trigger hallucinations).

**Self-RAG (Self-Reflective Retrieval-Augmented Generation)** trains an LLM to dynamically control its own retrieval and critique its own generations using **Special Reflection Tokens**:
1. **`[Retrieve]` Gating:** Output before generating:
$$\text{Retrieve} \in \{\text{[Retrieve=yes]}, \text{[Retrieve=no]}, \text{[Retrieve=continue]}\}$$
Only triggers external vector retrieval when necessary.

2. **`[IsRel]` Critique:** Evaluates retrieved passage relevance:
$$\text{IsRel} \in \{\text{[IsRel=relevant]}, \text{[IsRel=irrelevant]}\}$$

3. **`[IsSup]` Grounding:** Evaluates whether generated claim is supported by retrieved context:
$$\text{IsSup} \in \{\text{[IsSup=fully]}, \text{[IsSup=partially]}, \text{[IsSup=no]}\}$$

4. **`[IsUse]` Utility:** Evaluates overall response helpfulness:
$$\text{IsUse} \in \{1, 2, 3, 4, 5\}$$

---

### 2. Scratch Python Implementation: Adaptive Self-RAG Decision Engine

In [ ]:
import numpy as np

class MockSelfRAGSystem:
    def __init__(self):
        # Knowledge bank
        self.knowledge_base = {
            "paxos": "Paxos achieves consensus through Prepare/Promise and Propose/Accept phases.",
            "raft": "Raft is an understandable consensus algorithm using Leader, Follower, and Candidate states."
        }
        
    def evaluate_retrieval_need(self, prompt):
        """Simulates [Retrieve] gating token logits."""
        factual_keywords = ['consensus', 'paxos', 'raft', 'algorithm', 'formula']
        needs_retrieval = any(k in prompt.lower() for k in factual_keywords)
        prob = 0.95 if needs_retrieval else 0.05
        return (prob > 0.5), prob
        
    def evaluate_grounding(self, passage, generated_response):
        """Simulates [IsSup] reflection critique token."""
        words_in_passage = set(passage.lower().split())
        words_in_gen = set(generated_response.lower().split())
        overlap = len(words_in_passage.intersection(words_in_gen))
        
        if overlap > 4:
            return "[IsSup=Fully Supported]"
        elif overlap > 1:
            return "[IsSup=Partially Supported]"
        else:
            return "[IsSup=Not Supported - Hallucination]"

    def generate(self, user_prompt):
        print(f"User Query: '{user_prompt}'")
        needs_retrieve, p = self.evaluate_retrieval_need(user_prompt)
        
        if not needs_retrieve:
            print(f"  Decision: [Retrieve=No] (P={1-p:.2f}). Generating directly from parametric memory.")
            return "Parametric generation: Hello! How can I assist your coding today?"
            
        print(f"  Decision: [Retrieve=Yes] (P={p:.2f}). Initiating vector search.")
        # Retrieve context
        retrieved_doc = self.knowledge_base.get("raft", "General consensus data.")
        print(f"  Retrieved Passage: '{retrieved_doc}'")
        
        # Generation step
        candidate_ans = "Raft is an understandable consensus algorithm dividing roles into Leader and Candidate."
        grounding_critique = self.evaluate_grounding(retrieved_doc, candidate_ans)
        print(f"  Reflection Evaluation: {grounding_critique}")
        return candidate_ans

rag = MockSelfRAGSystem()
print("--- Query 1: Casual Prompt ---")
res1 = rag.generate("Hi, write a polite greeting.")

print("\n--- Query 2: Factual Systems Query ---")
res2 = rag.generate("Explain how the Raft consensus protocol handles leader state.")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Gating**: Evaluates retrieval need before querying vector databases.
2. **Grounding Critique**: Verifies factual support before delivering outputs.
3. **Controllable Output**: Rejects unsupported responses automatically.

---

### 4. Industry Applications & Trade-offs
- **Enterprise Customer Support Bots:** Avoids slow vector lookups for casual conversational pleasantries while grounding technical answers.
- **Trade-off:** Requires specialized fine-tuning with reflection tokens.

---

### 5. Top Interview Questions & Answers
**Q1: How does Self-RAG mitigate the retrieval latency penalty for simple queries?**
> **Answer:** By predicting the `[Retrieve=no]` token for casual or non-factual queries, bypassing vector index searches and generating directly from parametric model memory.

**Q2: What is the purpose of the `[IsSup]` token in Self-RAG?**
> **Answer:** It acts as an automated self-checker that evaluates whether the generated output is grounded in the retrieved text passages, detecting hallucinations prior to final generation.